# 04 - Investigating the tree-based attenuation

Three diagnostics for why tree-based DML attenuates relative to Lasso/OLS:
nuisance R² comparison, hyperparameter sensitivity, and overfitting check.

## 0. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('__file__').resolve().parent.parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from doubleml import DoubleMLData, DoubleMLPLR

from src.panel import build_sample, add_country_trends, weighted_demean_twoway

In [ ]:
# same data pipeline as notebook 03
panel = pd.read_csv('../data/processed/panel.csv')
panel = build_sample(panel)
df = panel[panel['col2']].copy().reset_index(drop=True)

df, trend_cols = add_country_trends(df)
BASE_CONTROLS = ['l1_indctryprod', 'l1_gdp', 'l1_taxsubs'] + trend_cols
Y_COL, D_COL, W_COL = 'lnrdexgov', 'l1_rdgov', 'weight'

w = df[W_COL].values.astype(float)
fe1, fe2 = df['indctry'].values, df['indyear'].values

cols_to_demean = [Y_COL, D_COL] + BASE_CONTROLS
mat = df[cols_to_demean].values.astype(float)
mat_dem = weighted_demean_twoway(mat, fe1, fe2, w)

dem_df = pd.DataFrame(mat_dem, columns=cols_to_demean)
Y_dem = dem_df[Y_COL].values
D_dem = dem_df[D_COL].values
X_dem = dem_df[BASE_CONTROLS].values

from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(dem_df[['l1_indctryprod', 'l1_gdp', 'l1_taxsubs']].values)
X_ml = np.hstack([X_poly, dem_df[trend_cols].values])

print(f'Sample: {len(df):,} obs, X_dem: {X_dem.shape}, X_ml: {X_ml.shape}')

## 1. Nuisance R² diagnostic

If trees absorb more treatment variation (D) than Lasso, less residual is left to identify tau.

In [ ]:
learners = {
    'Lasso': lambda: LassoCV(cv=3),
    'RF (NB03 defaults)': lambda: RandomForestRegressor(
        n_estimators=200, max_features='sqrt', min_samples_leaf=5, random_state=42
    ),
    'GBM': lambda: GradientBoostingRegressor(
        n_estimators=100, max_depth=3, random_state=42
    ),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_results = []

for name, make_learner in learners.items():
    # Linear features for Lasso, polynomial for trees
    X_feat = X_dem if 'Lasso' in name else X_ml

    for target_name, target in [('Y (outcome)', Y_dem), ('D (treatment)', D_dem)]:
        oof_preds = np.zeros_like(target)
        in_sample_r2s = []

        for train_idx, val_idx in kf.split(X_feat):
            model = make_learner()
            model.fit(X_feat[train_idx], target[train_idx])
            oof_preds[val_idx] = model.predict(X_feat[val_idx])
            in_sample_r2s.append(r2_score(target[train_idx], model.predict(X_feat[train_idx])))

        oof_r2 = r2_score(target, oof_preds)
        in_r2 = np.mean(in_sample_r2s)

        r2_results.append({
            'Learner': name,
            'Target': target_name,
            'OOF R²': oof_r2,
            'In-sample R²': in_r2,
            'Gap (overfit)': in_r2 - oof_r2,
        })

r2_df = pd.DataFrame(r2_results)
print(r2_df.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, target in zip(axes, ['Y (outcome)', 'D (treatment)']):
    sub = r2_df[r2_df['Target'] == target]
    x = np.arange(len(sub))
    width = 0.35

    ax.bar(x - width/2, sub['OOF R²'], width, label='Out-of-fold', color='steelblue', alpha=0.85)
    ax.bar(x + width/2, sub['In-sample R²'], width, label='In-sample', color='coral', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(sub['Learner'], fontsize=8)
    ax.set_ylabel('R²')
    ax.set_title(f'Nuisance fit: {target}')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)

fig.tight_layout()
Path('../output').mkdir(exist_ok=True)
fig.savefig('../output/fig_nuisance_r2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# how much treatment variation survives partialling out?
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"Demeaned D (before partialling): std = {D_dem.std():.4f}")
print()

for name, make_learner in learners.items():
    X_feat = X_dem if 'Lasso' in name else X_ml
    v_hat = np.zeros_like(D_dem)

    for train_idx, val_idx in kf.split(X_feat):
        m = make_learner()
        m.fit(X_feat[train_idx], D_dem[train_idx])
        v_hat[val_idx] = D_dem[val_idx] - m.predict(X_feat[val_idx])

    print(f'{name:25s}  std(v_hat) = {v_hat.std():.4f}  '
          f'({v_hat.std() / D_dem.std():.1%} of original)')

## 2. Hyperparameter sensitivity

Vary `n_estimators`, `max_depth`, `min_samples_leaf` to check if attenuation is robust to tuning.

In [ ]:
rf_configs = [
    {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 5},    # NB03 default
    {'n_estimators': 500, 'max_depth': None, 'min_samples_leaf': 5},
    {'n_estimators': 200, 'max_depth': 10,   'min_samples_leaf': 5},
    {'n_estimators': 200, 'max_depth': 20,   'min_samples_leaf': 5},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 1},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 20},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_leaf': 50},
]

dml_data_poly = DoubleMLData.from_arrays(x=X_ml, y=Y_dem, d=D_dem)
dml_data_lin  = DoubleMLData.from_arrays(x=X_dem, y=Y_dem, d=D_dem)

sensitivity_results = []

# lasso baseline
np.random.seed(42)
dml_base = DoubleMLPLR(dml_data_lin, ml_l=LassoCV(cv=5), ml_m=LassoCV(cv=5),
                        n_folds=5, n_rep=5, score='partialling out')
dml_base.fit()
sensitivity_results.append({'Config': 'Lasso (baseline)', 'tau': float(dml_base.coef[0])})
print(f'Lasso baseline: tau = {dml_base.coef[0]:.4f}\n')

# RF sweep
for i, cfg in enumerate(rf_configs):
    np.random.seed(42)
    label = f"RF(n={cfg['n_estimators']}, depth={cfg['max_depth']}, leaf={cfg['min_samples_leaf']})"
    dml = DoubleMLPLR(
        dml_data_poly,
        ml_l=RandomForestRegressor(**cfg, max_features='sqrt', random_state=42),
        ml_m=RandomForestRegressor(**cfg, max_features='sqrt', random_state=42),
        n_folds=5, n_rep=5, score='partialling out',
    )
    dml.fit()
    tau = float(dml.coef[0])
    default_tag = '  <- NB03 default' if i == 1 else ''
    sensitivity_results.append({'Config': label, 'tau': tau})
    print(f'[{i+1}/{len(rf_configs)}] {label:50s} tau = {tau:.4f}{default_tag}')

sens_df = pd.DataFrame(sensitivity_results)
print('\n' + sens_df.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

y_pos = list(range(len(sens_df)))[::-1]
for i, (_, row) in enumerate(sens_df.iterrows()):
    yi = y_pos[i]
    c = 'darkorange' if 'Lasso' in row['Config'] else 'steelblue'
    ax.plot(row['tau'], yi, 'o', color=c, ms=8, zorder=5)

ax.axvline(0.143, color='black', ls='--', lw=1, alpha=0.7, label='OLS (paper)')
ax.set_yticks(y_pos)
ax.set_yticklabels(sens_df['Config'], fontsize=8)
ax.set_xlabel('tau')
ax.set_title('RF hyperparameter sensitivity')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig('../output/fig_hp_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Overfitting diagnostic

In-sample vs out-of-fold R² gap: large gap on D suggests trees absorb causal signal.

In [ ]:
print('Overfitting gap = in-sample R² minus out-of-fold R²')
print()

for target in ['Y (outcome)', 'D (treatment)']:
    sub = r2_df[r2_df['Target'] == target]
    print(f'{target}:')
    for _, row in sub.iterrows():
        gap = row['Gap (overfit)']
        flag = '  << large gap' if gap > 0.15 else ''
        print(f"  {row['Learner']:25s}  gap = {gap:.4f}{flag}")
    print()

In [ ]:
# overfitting gap bar chart
fig, ax = plt.subplots(figsize=(9, 4.5))

targets = ['Y (outcome)', 'D (treatment)']
learner_names = r2_df['Learner'].unique()
x = np.arange(len(learner_names))
width = 0.35

for j, target in enumerate(targets):
    sub = r2_df[r2_df['Target'] == target]
    gaps = sub['Gap (overfit)'].values
    offset = (j - 0.5) * width
    colour = 'steelblue' if j == 0 else 'coral'
    ax.bar(x + offset, gaps, width, label=target, color=colour, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(learner_names, fontsize=9)
ax.set_ylabel('R² gap (in-sample - OOF)')
ax.set_title('Overfitting diagnostic')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig('../output/fig_overfit_gap.png', dpi=150, bbox_inches='tight')
plt.show()